# Predictive Maintenance — Machine Failure Classification

This notebook builds an end-to-end predictive maintenance pipeline: exploratory data analysis, feature engineering, class-imbalance handling, model benchmarking (Logistic Regression, Random Forest, XGBoost, LightGBM), hyperparameter tuning, threshold-aware evaluation, and model interpretability with SHAP.

The dataset contains 10,000 industrial sensor records with 5 failure modes (TWF, HDF, PWF, OSF, RNF) aggregated into a binary target `Machine failure`.


## 1. Environment & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

import shap
import joblib

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42

## 2. Data Loading

Place the source CSV file in the `data/` directory next to this notebook.

In [ ]:
DATA_PATH = "data/ai4i2020.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

## 3. Exploratory Data Analysis

In [ ]:
df["Machine failure"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
fig, ax = plt.subplots()
sns.countplot(data=df, x="Machine failure", ax=ax)
ax.set_title("Target Distribution — Machine Failure")
ax.set_xticklabels(["No Failure", "Failure"])
plt.show()

In [ ]:
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
df[failure_cols].sum().sort_values(ascending=False).plot(kind="bar", title="Failure Mode Frequency")
plt.ylabel("Count")
plt.show()

In [ ]:
numeric_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
df[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for col, ax in zip(numeric_cols, axes.flatten()):
    sns.histplot(data=df, x=col, hue="Machine failure", kde=True, ax=ax, element="step")
    ax.set_title(col)
axes.flatten()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
corr = df[numeric_cols + ["Machine failure"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Type", y="Torque [Nm]", hue="Machine failure")
plt.title("Torque by Product Type and Failure Status")
plt.show()

## 4. Feature Engineering

Domain-driven features derived from thermodynamics and mechanical power relationships known to drive the AI4I failure modes.

In [ ]:
data = df.copy()
data = data.drop(columns=["UDI", "Product ID"])

data["Temp_diff"] = data["Process temperature [K]"] - data["Air temperature [K]"]
data["Power"] = data["Torque [Nm]"] * data["Rotational speed [rpm]"] * (2 * np.pi / 60)
data["Torque_per_wear"] = data["Torque [Nm]"] / (data["Tool wear [min]"] + 1)
data["Speed_per_wear"] = data["Rotational speed [rpm]"] / (data["Tool wear [min]"] + 1)
data["Overstrain_index"] = data["Tool wear [min]"] * data["Torque [Nm]"]

le = LabelEncoder()
data["Type"] = le.fit_transform(data["Type"])

data.head()

In [ ]:
target = "Machine failure"
drop_leak = failure_cols

X = data.drop(columns=[target] + drop_leak)
y = data[target]

X.columns.tolist()

## 5. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train.shape, X_test.shape

## 6. Class Imbalance Handling

Machine failures represent roughly 3% of records. SMOTE oversampling is applied to the training split only, preventing information leakage into the test set.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

y_train.value_counts(), y_train_res.value_counts()

## 7. Baseline Model Benchmarking

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=300, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    preds = model.predict(X_test_scaled)
    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "F1": f1_score(y_test, preds),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
results_df

In [ ]:
results_df.set_index("Model")[["ROC-AUC", "PR-AUC", "F1"]].plot(kind="bar")
plt.title("Model Comparison")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

## 8. Hyperparameter Tuning — Best Model (XGBoost)

`GridSearchCV` with stratified 5-fold cross-validation, optimizing ROC-AUC.

In [ ]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)

grid.fit(X_train_res, y_train_res)
grid.best_params_, grid.best_score_

In [ ]:
best_model = grid.best_estimator_
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

## 9. Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Failure", "Failure"]).plot(cmap="Blues")
plt.title("Confusion Matrix — Tuned XGBoost")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

plt.plot(recall, precision, label=f"XGBoost (AP = {ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

## 10. Feature Importance

In [ ]:
importance = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.plot(kind="barh")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 11. Model Interpretability with SHAP

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, X_test_scaled, show=True)

In [ ]:
shap.summary_plot(shap_values, X_test_scaled, plot_type="bar", show=True)

## 12. Model Persistence

In [ ]:
joblib.dump(best_model, "predictive_maintenance_xgb.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")

## 13. Conclusion

- The engineered mechanical-power and thermal-differential features materially improved separability over raw sensor readings.
- SMOTE resampling on the training fold addressed the ~3% failure-class imbalance without leaking synthetic samples into evaluation.
- The tuned XGBoost classifier delivered the strongest ROC-AUC / PR-AUC trade-off among the benchmarked models.
- SHAP analysis confirms torque, tool wear, and the derived overstrain index as the dominant drivers of predicted failure risk.
